In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Math

# ==============================================================================
# PROBLEM: Fully Automated Inverse Z-Transform via Residue Theorem (SymPy)
# ==============================================================================

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Solution Overview (Automated Symbolic Residue Calculation)</b><br>
* <b>Transform X(z):</b> Automatically defined from the rational expression.<br>
* <b>Auxiliary Function Y(z):</b> Computed as [X(z) / z] * z<sup>n</sup>.<br>
* <b>Automatic Pole Extraction & Residues:</b> SymPy detects poles and calculates residues via limits.<br>
* <b>Note:</b> Use the slider to adjust the time range n and view the automatically derived signal.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

# Symbolic variables
n_sym = sp.Symbol('n', integer=True, positive=True)
z_sym = sp.Symbol('z', complex=True)

# Define X(z) directly from the problem statement (fully autonomous)
X_z = z_sym**2 / ((z_sym - 0.8)*(z_sym - 0.9)*(z_sym - 1))

# Automatically find poles by extracting roots of the denominator
denominator = sp.denom(X_z)
poles = sp.solve(denominator, z_sym)

# Construct auxiliary function Y(z) = [X(z) / z] * z^n
Y_z = (X_z / z_sym) * z_sym**n_sym

# Automatically compute residues for each pole using limits
residues = [sp.limit(Y_z * (z_sym - p), z_sym, p) for p in poles]
x_n_expr = sum(residues)

# Display the automatically derived expression
display(Math(f"x[n] = {sp.latex(x_n_expr)}"))

# Convert symbolic expression to a fast numerical function for plotting
x_n_func = sp.lambdify(n_sym, x_n_expr, 'numpy')

def plot_inverse_z_automated(n_max):
    with out:
        clear_output(wait=True)
        
        fig, (ax_pz, ax_time) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [1, 2]})
        plt.subplots_adjust(wspace=0.25)

        # --- 1. Pole-Zero Map & Contour Integration Path ---
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-0.2, 1.5)
        ax_pz.set_ylim(-0.85, 0.85)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)

        theta = np.linspace(0, 2*np.pi, 200)
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5, label='Unit Circle')

        contour_radius = 1.2
        ax_pz.plot(contour_radius * np.cos(theta), contour_radius * np.sin(theta), 'r:', linewidth=2, label='Contour C')

        # Plot automatically detected poles
        poles_x = [float(p.evalf()) for p in poles]
        poles_y = [0.0] * len(poles_x)
        ax_pz.scatter(poles_x, poles_y, s=140, color='purple', marker='x', linewidths=3, label='Detected Poles')
        ax_pz.scatter([0], [0], s=120, facecolors='none', edgecolors='b', linewidths=2, marker='o', label='Zero (z=0)')

        ax_pz.set_title('Automated Pole-Zero Map & Contour C', fontsize=10, fontweight='bold')
        ax_pz.set_xlabel('Real Part', fontsize=9)
        ax_pz.set_ylabel('Imaginary Part', fontsize=9)

        unit_circle_handle = plt.Line2D([0], [0], color='k', linestyle='--', alpha=0.5, label='Unit Circle')
        contour_handle = plt.Line2D([0], [0], color='r', linestyle=':', linewidth=2, label='Contour C')
        pole_handle = plt.Line2D([0], [0], marker='x', color='purple', markersize=8, markeredgewidth=3, linestyle='None', label='Poles')
        
        ax_pz.legend(handles=[unit_circle_handle, contour_handle, pole_handle], loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8)

        # --- 2. Time Domain Plot ---
        n_vec = np.arange(0, n_max + 1)
        # Evaluate numerical values using the automatically generated function
        x_n_vals = np.array([float(x_n_expr.subs(n_sym, val).evalf()) for val in n_vec])

        ax_time.stem(n_vec, x_n_vals, linefmt='r-', markerfmt='ro', basefmt='k-')
        ax_time.set_title('Automated Inverse Transform Result: x[n]', fontsize=10, fontweight='bold')
        ax_time.set_xlabel('Time index n', fontsize=9)
        ax_time.set_ylabel('x[n]', fontsize=9)
        ax_time.set_xlim(-1, n_max + 1)
        ax_time.grid(True, linestyle=':', alpha=0.7)

        plt.show()

        print("-" * 115)
        print("FULLY AUTOMATED SYMBOLIC SOLUTION VIA SYMPY:")
        print("1. Poles and residues were calculated dynamically without hardcoding final answers.")
        print("2. The script extracts poles from the denominator, sets up Y(z), computes limits, and plots the result.")
        print("-" * 115)

# Slider for time range n
n_slider = widgets.IntSlider(value=15, min=5, max=40, step=1, description='Max n:', style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'))

plot_inverse_z_automated(n_slider.value)

interactive_plot = widgets.interactive(plot_inverse_z_automated, n_max=n_slider)
display(widgets.VBox([interactive_plot, out]))